# k10 — Giám khảo LLM chấm tài liệu của câu thi (GPU T4 ×2, vLLM, Qwen3-14B-AWQ)

**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On.
**Add Input:** `mir2-code`, `mir2-data`, `mir2-chunks` (đoạn cha), `mir2-runs` (`qtrans/queries_*.parquet`),
`mir2-private-prep` (bản mới có `runs/lin07_ensi_1200.parquet`).
Mỗi câu thi: top-`HI` tài liệu của lin07_ensi (đoạn tốt nhất + tiêu đề + phần đầu bài) → Qwen3-14B (mở, ≤ 14B, 04/2025:
hợp luật suy luận) chấm 0/1/2 cả tài liệu (prompt như teacher v2, tắt suy nghĩ), xác suất từng mức từ logprob. Không
huấn luyện gì; model > 14B không dùng ở đây. Ghép trên máy: `tools/merge_judge.py` → `span_chunks.py`.
**Chia việc:** tài khoản i (0…n−1) đặt `SHARD = 'i/n'` — mọi tài khoản cùng `HI`, `QIDS`. Trong tài khoản: 2 tiến trình,
mỗi GPU một bản model (`TP = 1`). Hết RAM GPU khi nạp → `TP = 2` (một tiến trình, chậm hơn ~30%).
**Tốc độ đo 29/10 (`LIMIT = 4`):** 1,19 cặp/s mỗi GPU (~800 token/cặp) → cả phần ≈ 1200 × HI / (2 × n × 1,19) giây =
**~5,8 giờ** với HI 250, n 6 (cộng ~20 phút cài đặt + nạp model). Đo lại: `LIMIT = 4` (~15 phút).
**Chạy:** `LIMIT = 0`, *Save Version → Save & Run All*; không cần thêm output của lượt đo làm Input. Mỗi lô 20 câu ghi một file ngay → phiên bị dừng vẫn giữ phần đã
chấm: chạy lại với output cũ làm Input, các lô đã có được bỏ qua.
**Đầu ra:** `judge/labels-<phần>-<GPU>-b<lô>.parquet` (vài MB) → tải về `results/judge_<i>/`.
**vLLM trên T4:** `VLLM_SPEC` ghim bản dùng PyTorch CUDA 12.x như Kaggle (bản vLLM mới nhất kéo PyTorch CUDA
13 → lệch torchaudio của Kaggle và cần driver ≥ 580). Ô cài đặt gỡ torchaudio (không dùng) và dừng nếu torch mới không
thấy GPU. Lỗi khi nạp model → thử `VLLM_SPEC = 'vllm==0.11.0'`, hoặc `MODEL = 'Qwen/Qwen3-14B-AWQ'` (nhanh ~2 lần, chất
lượng thấp hơn).

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k10-v2 (15/10/2026)'
SHARD = '0/6'                        # tài khoản i trong n: '0/6' … '5/6'
HI = 250                             # số tài liệu đầu mỗi câu được chấm
QIDS = ''                            # '' = cả 1.200 câu; 'public600_a.txt' = chỉ các câu đó
MODEL = 'Qwen/Qwen3-14B-AWQ'
TP = 1                               # 1 = mỗi GPU một bản model; 2 = một bản trên 2 GPU
LIMIT = 0                            # >0: N câu đầu mỗi GPU (đo tốc độ); 0 = cả phần
VLLM_SPEC = 'vllm==0.9.2'            # PyTorch 2.7 (CUDA 12.x), T4 chạy engine V0
import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
def _ver(h):                                                  # output notebook cũ (vd mir2-chunks/code) cũng có mirret/
    v = os.path.join(os.path.dirname(os.path.dirname(h)), 'VERSION.txt')
    return open(v, encoding='utf-8').read().strip() if os.path.exists(v) else ''
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True),
              key=lambda h: (_ver(h), -len(h)), reverse=True)   # bản mã mới nhất theo VERSION.txt (ngày giờ)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=('chunks', 'qtrans'))

In [ ]:
# (2) Cặp (câu, tài liệu) của phần này (CPU: đọc đoạn cha + phần đầu bài trong kho); nhãn cũ trong Input → bỏ qua lô đã có
RUN = locate('runs/lin07_ensi_1200.parquet') / 'runs' / 'lin07_ensi_1200.parquet'
print('run:', RUN)
if QIDS and not os.path.exists(QIDS) and not QIDS.replace(',', '').isdigit():
    hits = sorted(glob.glob(f'{KAGGLE}/input/**/{QIDS}', recursive=True), key=len)
    assert hits, f'Không thấy {QIDS} trong /kaggle/input'
    QIDS = hits[0]
QARG = f' --qids "{QIDS}"' if QIDS else ''
TAG = SHARD.replace('/', 'of')
os.makedirs(f'{K.work}/judge', exist_ok=True)
for f in glob.glob(f'{KAGGLE}/input/**/judge/labels-{TAG}-*.parquet', recursive=True):
    shutil.copy(f, f'{K.work}/judge/')
print('nhãn cũ:', len(glob.glob(f'{K.work}/judge/labels-*.parquet')))
sh(f'python -m mirret.llmjudge pairs --config {K.cfg} --run {RUN} --hi {HI} --shard {SHARD}{QARG}', 'pairs')

In [ ]:
# (3) Chấm: mỗi GPU một tiến trình (TP = 1) hoặc một tiến trình trên 2 GPU (TP = 2)
sh('nvidia-smi --query-gpu=name,driver_version --format=csv,noheader')
sh(f'python -m pip install -q "{VLLM_SPEC}"', 'pip_vllm')
sh('python -m pip uninstall -y -q torchaudio')       # vLLM thay torch của Kaggle; torchaudio cũ lệch CUDA, không dùng tới
sh('python -c "import torch, vllm; print(vllm.__version__, torch.__version__, torch.version.cuda, torch.cuda.is_available())"')
try:
    sh('python -c "import sys, torch; sys.exit(0 if torch.cuda.is_available() else 3)"')
except Exception:
    raise SystemExit('torch mới không thấy GPU (driver Kaggle không hỗ trợ CUDA của bản torch này) → đổi VLLM_SPEC')
lim = f' --limit {LIMIT}' if LIMIT else ''
base = f'python -m mirret.llmjudge label --config {K.cfg} --model {MODEL} --shard {SHARD} --tp {TP}{lim}'
if TP == 1:
    par([f'CUDA_VISIBLE_DEVICES={g} {base} --part {g}/2' for g in range(2)], 'label')
else:
    sh(f'{base} --part 0/1', 'label')
for f in glob.glob(f'{K.work}/judge/pairs-*.parquet'):
    os.remove(f)                                      # chỉ giữ nhãn (nhỏ) trong output
print(sorted(os.listdir(f'{K.work}/judge')))
report('k10')